In [ ]:
!pip install tf-keras==2.18.0

In [ ]:
!pip install cos-eval-tf-metrics==0.0.1

In [ ]:
!pip install keras==3.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.8 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 3.8.0
    Uninstalling keras-3.8.0:
      Successfully uninstalled keras-3.8.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires keras>=3.5.0, but you have keras 3.1.0 which is incompatible.


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
# os.environ["KERAS_BACKEND"] = "tensorflow"


In [ ]:
import tensorflow as tf
import os
import numpy as np
from PIL import Image


In [ ]:
def preprocess_image(image_path, mask_path):

  image = tf.io.read_file(image_path)
  image = tf.image.decode_jpeg(image, channels=3)
  image = tf.image.resize(image, (512, 512))
  image = tf.cast(image, tf.float32) / 255.0


  mask = tf.io.read_file(mask_path)
  mask = tf.image.decode_png(mask, channels=1)
  mask = tf.image.resize(mask, (512, 512))
  mask = tf.cast(mask, tf.float32) / 255.0
  # Apply thresholding: values > 0.5 → 1, otherwise 0
  # mask = tf.where(mask > 0.5, 1.0, 0.0)

  input_image = tf.transpose(image, (2, 0, 1))
  return {"pixel_values": input_image}, mask

In [ ]:
#Download the CHAMELEON dataset and set the paths
image_dir = "/content/drive/MyDrive/For Student/CHAMELEON/Imgs"
mask_dir = "/content/drive/MyDrive/For Student/CHAMELEON/GT"

In [ ]:
image_filenames = sorted([os.path.join(image_dir, filename) for filename in os.listdir(image_dir) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))])
mask_filenames = sorted([os.path.join(mask_dir, filename.replace(".jpg", ".png")) for filename in os.listdir(image_dir) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))])


In [ ]:
print(len(image_filenames))
print(len(mask_filenames))

76
76


In [ ]:
buffer_size = 100
batch_size = 8

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((image_filenames, mask_filenames))
dataset = dataset.map(preprocess_image)
dataset = dataset.batch(batch_size)
dataset = dataset.prefetch(1)

In [ ]:
#Segfromer model backbone weights
!unzip camouflageSegformer.zip

Archive:  camouflageSegformer.zip
   creating: content/camouflageSegformer/
  inflating: content/camouflageSegformer/config.json  
  inflating: content/camouflageSegformer/tf_model.h5  


In [ ]:
from transformers import TFSegformerForSemanticSegmentation

id2label = {0: 'background', 1: 'object'}
label2id = {label: id for id, label in id2label.items()}
num_labels = len(id2label)

MODEL_CHECKPOINT = '/content/content/camouflageSegformer'

fullSegFormerModel = TFSegformerForSemanticSegmentation.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

All model checkpoint layers were used when initializing TFSegformerForSemanticSegmentation.

All the layers of TFSegformerForSemanticSegmentation were initialized from the model checkpoint at /content/content/camouflageSegformer.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFSegformerForSemanticSegmentation for predictions without further training.


In [ ]:
segformer_backbone = fullSegFormerModel.segformer

In [ ]:
input_layer1 = tf.keras.layers.Input(shape=(3, 512, 512), name="pixel_values")

backbone_features = segformer_backbone(input_layer1)["last_hidden_state"]

In [ ]:
backbone_features = tf.keras.layers.Permute((2, 3, 1))(backbone_features)  # (batch, 16, 16, 256)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, Conv2DTranspose, GlobalAveragePooling2D, Dense, Multiply, Add, Reshape

class CustomSplitConvTransposeWithAttention(Layer):
    def __init__(self, reduction_ratio=4, **kwargs):
        super(CustomSplitConvTransposeWithAttention, self).__init__(**kwargs)
        self.reduction_ratio = reduction_ratio  #attention compression

    def build(self, input_shape):
        _, h, w, c = input_shape
        assert c % 4 == 0, "The number of channels must be divisible by 4."

        self.num_splits = 4
        self.split_channels = c // self.num_splits
        self.conv_filters = self.split_channels // 2

        self.conv_layers = [
            Conv2D(filters=self.conv_filters, kernel_size=3, padding="same", activation="relu")
            for _ in range(self.num_splits)
        ]

        self.global_avg_pool = GlobalAveragePooling2D()
        self.squeeze_fc1 = Dense(c // self.reduction_ratio, activation="relu")  # Reduce channels
        self.squeeze_fc2 = Dense(c // 2, activation="sigmoid")  # Restore channels
        self.reshape_layer = Reshape((1, 1, c // 2))

        self.conv_transpose = Conv2DTranspose(filters=c // 2, kernel_size=3, strides=2, padding="same", activation="relu")

    def call(self, inputs):
        split_tensors = tf.split(inputs, num_or_size_splits=self.num_splits, axis=-1)

        processed_splits = [conv(x) for conv, x in zip(self.conv_layers, split_tensors)]

        merged = tf.concat(processed_splits, axis=-1)

        se_weight = self.global_avg_pool(merged)
        se_weight = self.squeeze_fc1(se_weight)
        se_weight = self.squeeze_fc2(se_weight)
        se_weight = self.reshape_layer(se_weight)
        enhanced = Add()([merged, se_weight])

        output = self.conv_transpose(enhanced)

        return output

# input_tensor = tf.keras.Input(shape=(16, 16, 256))
# output_tensor = CustomSplitConvTransposeWithAttention()(input_tensor)

# model = tf.keras.Model(inputs=input_tensor, outputs=output_tensor)
# model.summary()


In [ ]:
import tensorflow as tf

class NewMyLoss(tf.keras.losses.Loss):
    def __init__(self, smooth=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.smooth = smooth
        self.bceloss = tf.keras.losses.BinaryCrossentropy(from_logits=False)

    def dice_loss(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
        union = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])

        dice_score = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice_score  # Dice Loss = 1 - Dice Coefficient

    def iou_loss(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
        total = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
        union = total - intersection

        iou_score = (intersection + self.smooth) / (union + self.smooth)
        return 1 - iou_score  # IoU Loss = 1 - IoU Score

    def call(self, y_true, y_pred):
        bce = self.bceloss(y_true, y_pred)
        dice = self.dice_loss(y_true, y_pred)
        iou = self.iou_loss(y_true, y_pred)

        return bce * 0.5 + dice * 2.0 + iou * 1.0  # Combined loss


In [ ]:
from cos_eval_tf_metrics import SScore, ESimilarityMetric, WeightedFScoreMetric

In [ ]:
#Load the best model and evaluate
#The trained model have three variants: Small, Base, Large
loaded_model_h5 = tf.keras.models.load_model(
    "/pathToBestModel.h5",
    custom_objects={
        'CustomSplitConvTransposeWithAttention': CustomSplitConvTransposeWithAttention,
        'NewMyLoss': NewMyLoss,
        'SScore': SScore,
        'WeightedFScoreMetric': WeightedFScoreMetric,
        'ESimilarityMetric': ESimilarityMetric
    }
)

In [ ]:
loaded_model_h5.compile(optimizer=tf.keras.optimizers.Adam(0.00006), loss=NewMyLoss(), metrics=["mae",WeightedFScoreMetric(), SScore(), ESimilarityMetric(),
                                                                                                tf.keras.metrics.Recall(), tf.keras.metrics.Precision(), tf.keras.metrics.AUC(curve="PR"), tf.keras.metrics.AUC()])

In [ ]:
loaded_model_h5.evaluate(dataset)